# Module 9: RAG & LLM Applications
## 02: LLM API Fundamentals & Parameter Mastery

### Theoretical Overview
LLM APIs communicate over HTTP/HTTPS using standardized REST payloads. Understanding decoding parameters is crucial for controlling model behavior:
- **Temperature ($T$)**: Controls the sharpness of the probability distribution over tokens. $T \to 0$ leads to greedy deterministic selection; higher $T$ increases diversity.
- **Top-P (Nucleus Sampling)**: Dynamically truncates the probability distribution to the smallest set of tokens whose cumulative probability exceeds threshold $P$.
- **Frequency & Presence Penalties**: Discourage repetitive phrasing by penalizing tokens based on prior occurrences.

In [ ]:
import numpy as np

def simulate_softmax_sampling(logits: np.ndarray, temperature: float = 1.0, top_p: float = 0.9):
    # Scale logits by temperature
    scaled_logits = logits / max(temperature, 1e-5)
    # Numerical stability shift
    exp_logits = np.exp(scaled_logits - np.max(scaled_logits))
    probs = exp_logits / np.sum(exp_logits)
    
    # Sort probabilities descending
    sorted_indices = np.argsort(probs)[::-1]
    sorted_probs = probs[sorted_indices]
    
    # Cumulative probability mask for Top-P
    cumulative_probs = np.cumsum(sorted_probs)
    cutoff = cumulative_probs > top_p
    # Keep at least the top token
    cutoff[0] = False
    sorted_probs[cutoff] = 0.0
    sorted_probs /= np.sum(sorted_probs)
    
    return sorted_indices, sorted_probs

test_logits = np.array([4.2, 3.8, 2.1, 1.5, 0.2, -1.0])
indices, filtered_probs = simulate_softmax_sampling(test_logits, temperature=0.7, top_p=0.85)
for idx, p in zip(indices[:4], filtered_probs[:4]):
    print(f"Token Index: {idx:2d} | Probability: {p:.4f}")

### Standard OpenAI-Compatible Request Payload Anatomy

```json
{
  "model": "gpt-4o",
  "messages": [
    {"role": "system", "content": "You are a professional financial analyst assistant."},
    {"role": "user", "content": "Explain EBITDA to a non-technical executive."}
  ],
  "temperature": 0.3,
  "max_tokens": 512,
  "top_p": 0.95,
  "presence_penalty": 0.1,
  "frequency_penalty": 0.1
}
```

In [ ]:
def construct_chat_payload(system_prompt: str, user_prompt: str, model="gpt-4o", temp=0.2):
    return {
        "model": model,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        "temperature": temp,
        "response_format": {"type": "json_object"}
    }

payload = construct_chat_payload("You are a data extraction engine.", "Extract customer name and email from text.")
print(json.dumps(payload, indent=2))